# brier tutorial: a calibrated ticket router from an open LLM

You will turn **Qwen3-1.7B** into a router that sends customer messages to one of five teams,
with probabilities you can trust, and without fine-tuning or text generation:

1. check that the model works with brier (`brier check`);
2. ask a typed question with **no labels** (levels `raw` and `L0`);
3. calibrate with **150 labelled messages** (levels `L1` and `L2`) and compare;
4. save the calibration and reuse it, and route only when the model is confident.

**Runtime:** free Colab **T4** GPU (*Runtime -> Change runtime type -> T4 GPU*), about 15 min.
Data: 750 messages from [BANKING77](https://github.com/PolyAI-LDN/task-specific-datasets)
(CC-BY-4.0). With your own data you only need lists of strings and labels.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRIER_VERSION = "v0.1.0"  # a release tag, or "main" for the latest code
!pip install -q "brier[hf] @ git+https://github.com/mohdUwaish59/brier@{BRIER_VERSION}"

## 1. Does the model work with brier?

`brier check` loads the model and tests label tokens, batching and hidden-state access, and
runs a 14-item sanity task. Always pin `--revision` to a commit: a model name alone does not
pin the weights. We use float32: the T4 has no bfloat16, and float16 can overflow in some
models (the 1.7B model fits in float32).

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
!brier check {MODEL} --revision {REVISION} --dtype float32

## 2. The data: customer messages and their team

Five BANKING77 intents, each mapped to a team name. Per team: 30 labelled messages (for
calibration), 20 unlabelled ones (for the zero-label prior) and 30 for testing.

In [ ]:
import random
from collections import defaultdict

from brier.bench.tasks import fetch_banking77, parse_csv  # pinned, SHA-256 verified download

TEAMS = {
    "card_arrival": "card delivery",
    "lost_or_stolen_card": "lost or stolen card",
    "exchange_rate": "exchange rates",
    "top_up_failed": "failed top-up",
    "request_refund": "refunds",
}
by_team = defaultdict(list)
for text, intent in parse_csv(fetch_banking77()):
    if intent in TEAMS:
        by_team[TEAMS[intent]].append(text)

rng = random.Random(0)  # noqa: S311 - shuffling data, not security
labelled, unlabelled, test = [], [], []
for team, texts in sorted(by_team.items()):
    rng.shuffle(texts)
    labelled += [(t, team) for t in texts[:30]]
    unlabelled += texts[30:50]
    test += [(t, team) for t in texts[50:80]]
rng.shuffle(labelled)
print(len(labelled), "labelled /", len(unlabelled), "unlabelled /", len(test), "test")
print(test[0])

## 3. Ask the question with no labels: `raw` and `L0`

A `Choice` question; brier reads the model's probability for each option letter directly.
`raw` is a single read. `L0` reads the question once per rotation of the option order and
averages, which removes the model's preference for particular positions.

In [ ]:
import numpy as np

from brier import Choice, Decider
from brier.backends.hf import HFBackend
from brier.metrics import accuracy, ece

backend = HFBackend(MODEL, revision=REVISION, dtype="float32", batch_size=32)
d = Decider(backend, max_batch=256)  # default limit is 64 states per call
route = Choice("Which team should handle this message?", sorted(TEAMS.values()), name="route")

states = [t for t, _ in test]
y = [route.options.index(team) for _, team in test]


def evaluate(level):
    results = d.decide_batch(states, [route], level=level)
    probs = np.array([[r["route"].probs[o] for o in route.options] for r in results])
    return accuracy(probs, y), ece(probs, y)


scores = {}
for level in ("raw", "L0"):
    scores[level] = evaluate(level)
    print(
        f"{level:>3}: accuracy {scores[level][0]:.1%}, calibration error (ECE) {scores[level][1]:.3f}"
    )

`L0` also takes a pool of **unlabelled** messages to correct the model's bias towards some
options (a "batch prior"). Still no labels:

In [ ]:
d.fit_prior(unlabelled, [route])
scores["L0 + prior"] = evaluate("L0")
print("L0 + prior: accuracy {:.1%}, ECE {:.3f}".format(*scores["L0 + prior"]))

## 4. Calibrate with 150 labels: `L1` and `L2`

* `L1` fits one temperature so that confidence matches accuracy (answers do not change).
* `L2` trains a small linear head on the model's hidden states (seconds on a CPU; the LLM
  stays frozen).

Labels help most when the question is hard for the model. On these five clearly different teams
the model may already be accurate and well calibrated after `L0`, and `L1`/`L2` may not beat
it; on 20 banking intents, `L2` lifted Qwen3-1.7B from 67 % to 81 % accuracy and cut the
calibration error from 0.094 to 0.035
([results](https://github.com/mohdUwaish59/brier/blob/main/docs/results.md)). Always compare
levels on held-out data, as below, and use the one that does best on yours.

Labels are the option text, exactly as `Decision.answer` returns it.

In [ ]:
lab_states = [t for t, _ in labelled]
lab_answers = [team for _, team in labelled]

d.fit_temperature(lab_states, route, lab_answers)
scores["L1"] = evaluate("L1")
d.fit_head(lab_states, route, lab_answers)
scores["L2"] = evaluate("L2")

print("| level | accuracy | ECE (lower is better) |")
print("|---|---|---|")
for level, (acc, err) in scores.items():
    print(f"| {level} | {acc:.1%} | {err:.3f} |")

## 5. One decision, in full

Every decision carries the probabilities of all options, the answer, its confidence and the
level that produced it.

In [ ]:
msg = "I topped up with my card yesterday but the money never showed up in my account."
dec = d.decide(msg, [route], level="L2")["route"]
print("answer:", dec.answer, "| confidence:", round(dec.confidence, 3), "| level:", dec.level)
for option, p in sorted(dec.probs.items(), key=lambda kv: -kv[1]):
    print(f"  {p:6.1%}  {option}")

## 6. Route only when confident

Calibrated probabilities make a confidence threshold meaningful: send confident decisions
automatically and the rest to a human.

In [ ]:
results = d.decide_batch(states, [route], level="L2")
for threshold in (0.5, 0.7, 0.9):
    sure = [
        (r["route"].answer, team)
        for r, (_, team) in zip(results, test, strict=True)
        if r["route"].confidence >= threshold
    ]
    if not sure:
        print(f"confidence >= {threshold}: no message is that confident")
        continue
    acc = np.mean([a == t for a, t in sure])
    print(
        f"confidence >= {threshold}: routes {len(sure) / len(test):.0%} of messages "
        f"automatically, {acc:.1%} of them correctly"
    )

## 7. Save the calibration and reuse it

A calibration is a small folder (JSON + `.npz`, never model weights). Loading it checks that
the model, revision, precision and prompt templates match the ones it was fitted with.

In [ ]:
d.save("router_calibration")
d2 = Decider.load("router_calibration", backend, max_batch=256)
same = d2.decide(msg, [route], level="L2")["route"].probs == dec.probs
print("reloaded calibration gives identical decisions:", same)

## Next steps

* Use your own model: run `brier check <model> --revision <sha>` first.
* Use your own questions: `Choice`, `Noul` (yes/no, `p_yes`) and `Score` (ratings, `expected`).
* Read how each level works in
  [METHODS.md](https://github.com/mohdUwaish59/brier/blob/main/docs/METHODS.md) and the
  benchmark in [results.md](https://github.com/mohdUwaish59/brier/blob/main/docs/results.md).
* Questions, bugs and ideas:
  [GitHub issues](https://github.com/mohdUwaish59/brier/issues).